# Notebook 5: Ensemble Methods Overview

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2 hours | **Week 9 — Tree-Based Methods & Gradient Boosting**

---

## Why This Matters

Single models — no matter how carefully tuned — have fundamental limitations. A decision tree overfits. A linear model underfits complex patterns. Ensemble methods are the reason that **every major ML competition winner** for structured/tabular data uses some form of ensembling. Understanding *why* ensembles work (not just that they work) will let you diagnose failures and design better pipelines.

---

## The Expert Panel Analogy

Imagine you need to estimate the price of a house. You could:

- Ask **one expert**: fast and cheap, but that expert might have a blind spot (e.g., they undervalue gardens, they overvalue proximity to a train station).
- Ask **100 different experts** and average their answers: each expert has *different* blind spots. When you average, the random errors cancel out — one expert's over-estimate cancels another's under-estimate — and the systematic insights survive.

This is ensemble learning in one sentence: **many imperfect models combined beat one perfect-seeming model**, because their errors are independent.

The key word is *different*. If all 100 experts trained at the same school, read the same textbook, and use the same formula — their errors are the same, and averaging does nothing. **Diversity is the engine of ensemble power.**

## Three Ensemble Paradigms

| Paradigm | Training Order | How Predictions Combined | Primary Effect |
|---|---|---|---|
| **Bagging** | Parallel (independent) | Average / majority vote | Reduces **variance** |
| **Boosting** | Sequential (each fixes previous errors) | Weighted sum | Reduces **bias** |
| **Stacking** | Parallel base models, then meta-learner | Meta-learner decides weights | Can reduce both |

### 1. Bagging (Bootstrap Aggregating)
- Draw B bootstrap samples (random samples **with replacement**) from the training data
- Train one model on each bootstrap sample — independently, in parallel
- Prediction = average of all B models (regression) or majority vote (classification)
- **Why it works:** Each model sees a slightly different dataset → slightly different errors. Averaging cancels the random component of those errors (variance). The systematic component (bias) is unchanged.

### 2. Boosting
- Train model 1 on the original data
- Train model 2 on data where the *mistakes* of model 1 are up-weighted
- Train model 3 on data where the *mistakes* of models 1+2 are up-weighted… and so on
- Final prediction = weighted sum of all models
- **Why it works:** Each model fills in the gaps left by previous models → the ensemble's bias decreases with each round
- Examples: AdaBoost, Gradient Boosting, XGBoost, LightGBM, CatBoost

### 3. Stacking (Stacked Generalization)
- **Level 0:** Train K diverse base models (e.g., decision tree + ridge regression + k-NN)
- **Level 1:** Use the base models' **out-of-fold predictions** as features to train a meta-learner
- Meta-learner learns: "When the tree says X, the ridge says Y, and the k-NN says Z — what is the true answer?"
- Base models can be completely different families (no constraint)

## Bias-Variance Perspective

Recall the decomposition:
$$\text{Expected Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

**Bagging:** If B models each have variance $\sigma^2$ and their predictions have pairwise correlation $\rho$:
$$\text{Variance of ensemble} = \rho \sigma^2 + \frac{(1-\rho)\sigma^2}{B}$$

- As $B \to \infty$: variance $\to \rho \sigma^2$ (the "floor" is set by correlation, not number of models)
- If $\rho = 0$ (uncorrelated): variance $\to 0$ as $B \to \infty$
- If $\rho = 1$ (identical models): variance stays at $\sigma^2$ — averaging does nothing
- **Bootstrap sampling creates diversity** → lower $\rho$ → more benefit from averaging

**Boosting:** Each round fits a new model to the *residuals* of the previous ensemble. This directly attacks the bias (underfitting). However, with too many rounds, the ensemble can start memorizing noise → variance increases. This is why boosting can overfit with too many rounds (unlike bagging).

**Stacking:** Combines the outputs of diverse model families. If base models have uncorrelated errors (different inductive biases), the meta-learner can exploit this and reduce both bias and variance.

In [ ]:
# ============================================================
# Cell 1: Imports and Global Setup
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

np.random.seed(42)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
print('Libraries loaded. NumPy seed set to 42.')

In [ ]:
# ============================================================
# Cell 2: Synthetic House Price Dataset (600 samples, 6 features)
# ============================================================
np.random.seed(42)
n_samples = 600

# Feature generation
sqft         = np.random.normal(1800, 500, n_samples).clip(500, 4000)
bedrooms     = np.random.randint(1, 6, n_samples).astype(float)
age          = np.random.uniform(0, 50, n_samples)
distance_cbd = np.random.uniform(1, 30, n_samples)  # km from CBD
garage       = np.random.randint(0, 3, n_samples).astype(float)
school_score = np.random.uniform(4, 10, n_samples)  # local school rating

# House price formula with nonlinearities + noise
price = (
      80 * sqft
    + 15000 * bedrooms
    - 1200 * age
    - 5000 * distance_cbd
    + 18000 * garage
    + 12000 * school_score
    + 0.02 * sqft**2              # nonlinear sqft effect
    - 300 * (age * distance_cbd)  # interaction
    + np.random.normal(0, 25000, n_samples)  # noise
) / 1000  # scale to $k

price = price.clip(80, 900)  # realistic range

# Assemble DataFrame
df = pd.DataFrame({
    'sqft': sqft,
    'bedrooms': bedrooms,
    'age': age,
    'distance_cbd': distance_cbd,
    'garage': garage,
    'school_score': school_score,
    'price_k': price
})

# Binary target: above/below median price
median_price = df['price_k'].median()
df['above_median'] = (df['price_k'] > median_price).astype(int)

print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Price range: ${df.price_k.min():.0f}k – ${df.price_k.max():.0f}k')
print(f'Median price: ${median_price:.0f}k')
print(f'Above median: {df.above_median.sum()} ({100*df.above_median.mean():.1f}%)')
df.head()

In [ ]:
# ============================================================
# Cell 3: Train/Test Split
# ============================================================
FEATURES = ['sqft', 'bedrooms', 'age', 'distance_cbd', 'garage', 'school_score']
TARGET = 'price_k'

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Features:     {FEATURES}')

In [ ]:
# ============================================================
# Cell 4: Manual Bagging Implementation
# ============================================================
def manual_bagging(model_class, X_train, y_train, X_test,
                   n_estimators=10, random_state=42, **model_kwargs):
    """
    Bootstrap aggregating from scratch.
    - Draws n_estimators bootstrap samples
    - Trains model_class on each
    - Returns averaged predictions
    """
    np.random.seed(random_state)
    predictions = []
    for i in range(n_estimators):
        # Bootstrap: sample WITH replacement
        bootstrap_idx = np.random.choice(len(X_train), len(X_train), replace=True)
        X_boot = X_train[bootstrap_idx]
        y_boot = y_train[bootstrap_idx]
        # Fit model on this bootstrap sample
        model = model_class(**model_kwargs)
        model.fit(X_boot, y_boot)
        predictions.append(model.predict(X_test))
    # Average predictions across all estimators
    return np.mean(predictions, axis=0)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# --- Comparison: single tree vs manual bagging 10 trees vs 100 trees ---
# Single tree
single_tree = DecisionTreeRegressor(max_depth=None, random_state=RANDOM_STATE)
single_tree.fit(X_train, y_train)
rmse_single = rmse(y_test, single_tree.predict(X_test))

# Manual bagging: 10 trees
pred_bag10 = manual_bagging(DecisionTreeRegressor, X_train, y_train, X_test,
                             n_estimators=10, random_state=RANDOM_STATE,
                             max_depth=None)
rmse_bag10 = rmse(y_test, pred_bag10)

# Manual bagging: 100 trees
pred_bag100 = manual_bagging(DecisionTreeRegressor, X_train, y_train, X_test,
                              n_estimators=100, random_state=RANDOM_STATE,
                              max_depth=None)
rmse_bag100 = rmse(y_test, pred_bag100)

print('RMSE Comparison (lower is better):')
print(f'  Single decision tree:     {rmse_single:.2f} $k')
print(f'  Manual bagging (10 trees): {rmse_bag10:.2f} $k')
print(f'  Manual bagging (100 trees): {rmse_bag100:.2f} $k')
print(f'\nImprovement (100 trees vs single): {100*(rmse_single - rmse_bag100)/rmse_single:.1f}%')

In [ ]:
# ============================================================
# Cell 5: Visualisation — Single Tree vs Bagging Comparison
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Ensemble Effect on House Price Predictions\n(Sorted by True Price)',
             fontsize=14, fontweight='bold')

sort_idx = np.argsort(y_test)
x_axis = np.arange(len(y_test))

configs = [
    (single_tree.predict(X_test), f'Single Tree\nRMSE = {rmse_single:.1f}$k', '#e74c3c'),
    (pred_bag10,  f'Bagging 10 Trees\nRMSE = {rmse_bag10:.1f}$k',  '#e67e22'),
    (pred_bag100, f'Bagging 100 Trees\nRMSE = {rmse_bag100:.1f}$k', '#27ae60'),
]

for ax, (preds, title, color) in zip(axes, configs):
    ax.scatter(x_axis, y_test[sort_idx], s=12, alpha=0.5,
               color='#2c3e50', label='True price', zorder=3)
    ax.scatter(x_axis, preds[sort_idx], s=12, alpha=0.5,
               color=color, label='Predicted', zorder=2)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('House (sorted by true price)')
    ax.set_ylabel('Price ($k)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Note: Bagging predictions (right panel) are smoother — less variance.')

In [ ]:
# ============================================================
# Cell 6: Variance Reduction Demo
#   Run single tree 50 times with different seeds → large spread
#   Run manual bagging 50 times with different seeds → small spread
# ============================================================
N_TRIALS = 50

single_tree_preds = []
bagging_preds     = []

for seed in range(N_TRIALS):
    # Resplit data differently each trial
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)

    # Single tree prediction on ONE fixed test house (house index 0 in original data)
    t = DecisionTreeRegressor(max_depth=None, random_state=seed)
    t.fit(X_tr, y_tr)
    single_tree_preds.append(t.predict(X_te[:1])[0])

    # Bagging prediction on the same house
    bag_pred = manual_bagging(DecisionTreeRegressor, X_tr, y_tr, X_te[:1],
                              n_estimators=50, random_state=seed, max_depth=None)
    bagging_preds.append(bag_pred[0])

single_std = np.std(single_tree_preds)
bagging_std = np.std(bagging_preds)

print('Variance Reduction Demonstration:')
print(f'  Single tree prediction std (50 trials): ${single_std:.1f}k')
print(f'  Bagging prediction std   (50 trials): ${bagging_std:.1f}k')
print(f'  Variance reduction: {100*(1 - bagging_std/single_std):.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Prediction Variance: Single Tree vs Bagging (50 Trials)',
             fontsize=13, fontweight='bold')

axes[0].hist(single_tree_preds, bins=20, color='#e74c3c', alpha=0.75, edgecolor='white')
axes[0].axvline(np.mean(single_tree_preds), color='black', lw=2, linestyle='--', label=f'Mean = {np.mean(single_tree_preds):.0f}k')
axes[0].set_title(f'Single Tree\nStd = ${single_std:.1f}k', fontsize=11)
axes[0].set_xlabel('Predicted Price ($k)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].hist(bagging_preds, bins=20, color='#27ae60', alpha=0.75, edgecolor='white')
axes[1].axvline(np.mean(bagging_preds), color='black', lw=2, linestyle='--', label=f'Mean = {np.mean(bagging_preds):.0f}k')
axes[1].set_title(f'Bagging (50 trees)\nStd = ${bagging_std:.1f}k', fontsize=11)
axes[1].set_xlabel('Predicted Price ($k)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()
print('The bagging distribution is much tighter — lower variance, more reliable predictions.')

In [ ]:
# ============================================================
# Cell 7: Simple Stacking from Scratch
#   Base models: Decision Tree, Ridge, k-NN
#   Meta-learner: Ridge
#   Uses out-of-fold predictions to avoid data leakage
# ============================================================
from sklearn.model_selection import KFold

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Define base models
base_models = [
    ('DecisionTree', DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE)),
    ('Ridge',        Ridge(alpha=10.0)),
    ('k-NN',         KNeighborsRegressor(n_neighbors=10)),
]

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
n_train = X_train_scaled.shape[0]
n_base  = len(base_models)

# Level-0: Out-of-fold predictions (training meta-features)
oof_preds  = np.zeros((n_train, n_base))   # shape: (480, 3)
# Level-0: Full test set predictions (averaged over folds)
test_preds = np.zeros((X_test_scaled.shape[0], n_base))  # shape: (120, 3)

for fold_i, (fold_train_idx, fold_val_idx) in enumerate(kf.split(X_train_scaled)):
    X_fold_tr, X_fold_val = X_train_scaled[fold_train_idx], X_train_scaled[fold_val_idx]
    y_fold_tr, y_fold_val = y_train[fold_train_idx], y_train[fold_val_idx]

    for m_i, (name, model) in enumerate(base_models):
        model_clone = type(model)(**model.get_params())
        model_clone.fit(X_fold_tr, y_fold_tr)
        oof_preds[fold_val_idx, m_i] = model_clone.predict(X_fold_val)
        test_preds[:, m_i] += model_clone.predict(X_test_scaled) / kf.n_splits

print('Out-of-fold predictions shape:', oof_preds.shape)
print('Test meta-features shape:     ', test_preds.shape)

# Level-1: Train meta-learner on OOF predictions
meta_learner = Ridge(alpha=1.0)
meta_learner.fit(oof_preds, y_train)
stacking_preds = meta_learner.predict(test_preds)

# Compare all methods
print('\n--- RMSE Comparison ---')
rmse_tree  = rmse(y_test, DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE).fit(X_train_scaled, y_train).predict(X_test_scaled))
rmse_ridge = rmse(y_test, Ridge(alpha=10.0).fit(X_train_scaled, y_train).predict(X_test_scaled))
rmse_knn   = rmse(y_test, KNeighborsRegressor(n_neighbors=10).fit(X_train_scaled, y_train).predict(X_test_scaled))
rmse_stack = rmse(y_test, stacking_preds)

for name, r in [('Decision Tree (depth=5)', rmse_tree),
                ('Ridge', rmse_ridge),
                ('k-NN (k=10)', rmse_knn),
                ('Stacking (meta=Ridge)', rmse_stack)]:
    print(f'  {name:<30} RMSE = {r:.2f}$k')

In [ ]:
# ============================================================
# Cell 8: Meta-learner Coefficients — What Did Stacking Learn?
# ============================================================
print('Meta-learner (Ridge) coefficients:')
for (name, _), coef in zip(base_models, meta_learner.coef_):
    print(f'  {name:<15}: {coef:.4f}')
print(f'  Intercept:      {meta_learner.intercept_:.4f}')
print()
print('Interpretation: coefficient = how much the meta-learner trusts each base model.')
print('The meta-learner learned to blend, not just average.')

# Bar chart of coefficients
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#3498db', '#e74c3c', '#2ecc71']
ax.bar([n for n,_ in base_models], meta_learner.coef_, color=colors, edgecolor='white', linewidth=1.2)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Meta-Learner Coefficients\n(How much stacking trusts each base model)', fontsize=12)
ax.set_ylabel('Coefficient')
ax.set_xlabel('Base Model')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 9: sklearn BaggingRegressor vs Manual Implementation
# ============================================================
# sklearn BaggingRegressor
bag_sklearn = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=None),
    n_estimators=100,
    max_samples=1.0,   # use all samples (bootstrap=True by default)
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
bag_sklearn.fit(X_train, y_train)
rmse_sklearn_bag = rmse(y_test, bag_sklearn.predict(X_test))

rmse_manual_bag = rmse(y_test, pred_bag100)  # from Cell 4

print('Comparison: Manual Bagging vs sklearn BaggingRegressor (100 trees):')
print(f'  Manual bagging RMSE:   {rmse_manual_bag:.2f}$k')
print(f'  sklearn BaggingRegressor RMSE: {rmse_sklearn_bag:.2f}$k')
print()
print('Small differences are due to different internal random streams.')
print('sklearn also parallelises with n_jobs=-1 (uses all CPU cores).')

# Prediction scatter: are they similar?
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(pred_bag100, bag_sklearn.predict(X_test), alpha=0.5, s=15, color='#8e44ad')
lo = min(pred_bag100.min(), bag_sklearn.predict(X_test).min())
hi = max(pred_bag100.max(), bag_sklearn.predict(X_test).max())
ax.plot([lo, hi], [lo, hi], 'k--', lw=1.5, label='Perfect agreement')
ax.set_xlabel('Manual Bagging (100 trees)')
ax.set_ylabel('sklearn BaggingRegressor (100 trees)')
ax.set_title('Manual vs sklearn Bagging\nPredictions are nearly identical', fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 10: Diversity Demonstration
#   Correlated models = no benefit from averaging
#   Uncorrelated models = large benefit
# ============================================================
print('=== Diversity is the Engine of Ensemble Power ===')
print()
print('Scenario A: 10 IDENTICAL trees (same data, same seed) — NO diversity')
print('Scenario B: 10 DIVERSE trees (different bootstrap samples) — diversity')
print()

# Scenario A: identical trees
identical_tree = DecisionTreeRegressor(max_depth=None, random_state=0)
identical_tree.fit(X_train, y_train)
identical_preds = np.array([identical_tree.predict(X_test)] * 10)
rmse_identical = rmse(y_test, identical_preds.mean(axis=0))

corr_identical = np.corrcoef(identical_preds).mean()

# Scenario B: diverse trees (different bootstrap samples)
np.random.seed(RANDOM_STATE)
diverse_preds = []
for seed in range(10):
    boot_idx = np.random.choice(len(X_train), len(X_train), replace=True)
    t = DecisionTreeRegressor(max_depth=None, random_state=seed)
    t.fit(X_train[boot_idx], y_train[boot_idx])
    diverse_preds.append(t.predict(X_test))
diverse_preds = np.array(diverse_preds)
rmse_diverse = rmse(y_test, diverse_preds.mean(axis=0))
corr_diverse = np.corrcoef(diverse_preds).mean()

print(f'Scenario A (identical trees):')
print(f'  Average pairwise correlation: {corr_identical:.4f}')
print(f'  Ensemble RMSE: {rmse_identical:.2f}$k  (same as single tree!)')
print()
print(f'Scenario B (diverse trees, bootstrap):')
print(f'  Average pairwise correlation: {corr_diverse:.4f}')
print(f'  Ensemble RMSE: {rmse_diverse:.2f}$k  (better!)')
print()
print('Conclusion: Lower correlation → greater benefit from averaging.')
print('DIVERSITY IS ESSENTIAL. Averaging clones achieves nothing.')

# Visualise correlation matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Pairwise Prediction Correlations Between 10 Trees', fontsize=13, fontweight='bold')

for ax, preds, title in [
    (axes[0], identical_preds, f'Identical Trees\nMean Corr = {corr_identical:.3f}'),
    (axes[1], diverse_preds,   f'Diverse Trees (bootstrap)\nMean Corr = {corr_diverse:.3f}')
]:
    cm = np.corrcoef(preds)
    sns.heatmap(cm, ax=ax, vmin=0, vmax=1, cmap='RdYlGn_r',
                annot=True, fmt='.2f', linewidths=0.5, square=True, cbar_kws={'shrink': 0.7})
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Tree index')
    ax.set_ylabel('Tree index')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 11: Ensemble Learning Curve — n_estimators from 1 to 200
#   Shows where improvement plateaus
# ============================================================
from sklearn.model_selection import cross_val_score

estimator_counts = list(range(1, 11)) + list(range(15, 51, 5)) + list(range(60, 201, 20))
val_rmses = []

for n_est in estimator_counts:
    bag = BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=None),
        n_estimators=n_est,
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    # Use 5-fold CV RMSE as validation metric
    scores = cross_val_score(bag, X_train, y_train,
                              cv=3, scoring='neg_root_mean_squared_error')
    val_rmses.append(-scores.mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(estimator_counts, val_rmses, marker='o', markersize=4,
        color='#2980b9', linewidth=2, label='CV RMSE')
ax.axhline(val_rmses[-1], color='#e74c3c', linestyle='--', alpha=0.7, label=f'Plateau ≈ {val_rmses[-1]:.1f}$k')

# Mark the elbow
improvement = np.diff(val_rmses)
elbow_i = np.argmin(improvement[5:]) + 5  # ignore first 5
ax.axvline(estimator_counts[elbow_i], color='#27ae60', linestyle=':', linewidth=2,
           label=f'Elbow ≈ {estimator_counts[elbow_i]} trees')

ax.set_xlabel('Number of Estimators (n_estimators)', fontsize=12)
ax.set_ylabel('Validation RMSE ($k)', fontsize=12)
ax.set_title('Ensemble Learning Curve\nValidation RMSE vs Number of Trees', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

print(f'Key observation: RMSE drops steeply at first, then plateaus around {estimator_counts[elbow_i]} trees.')
print('Beyond that, adding more trees costs computation but buys little accuracy.')
print('Unlike boosting, more trees in bagging NEVER increases RMSE (it cannot overfit by adding trees).')

In [ ]:
# ============================================================
# Cell 12: Summary Table — Bagging vs Boosting vs Stacking
# ============================================================
summary_data = {
    'Paradigm':          ['Bagging',            'Boosting',             'Stacking'],
    'Training Order':    ['Parallel',            'Sequential',           'Parallel (base) → Sequential (meta)'],
    'Primary Effect':    ['Reduces Variance',    'Reduces Bias',         'Can reduce both'],
    'Overfits w/ more?': ['No (safe to add)',    'Yes (use early stop)', 'Depends on meta-learner'],
    'Parallelisable':    ['Yes',                 'No',                   'Partially'],
    'Interpretability':  ['Low',                 'Low',                  'Very Low'],
    'Computation':       ['Moderate',            'Moderate–High',        'High'],
    'Examples':          ['Random Forest',       'XGBoost, LightGBM',    'StackingRegressor'],
    'Best when':         ['High variance model', 'High bias model',      'Diverse model pool'],
}

df_summary = pd.DataFrame(summary_data)
print('=== Ensemble Methods Summary ===')
print(df_summary.to_string(index=False))
df_summary

In [ ]:
# ============================================================
# Cell 13: Bias-Variance Formula Visualisation
#   Ensemble variance = rho*sigma^2 + (1-rho)*sigma^2/B
# ============================================================
B_values  = np.arange(1, 201)
sigma2    = 1.0  # single-tree variance (normalised)

fig, ax = plt.subplots(figsize=(10, 5))
colors_rho = ['#2c3e50', '#2980b9', '#27ae60', '#e67e22', '#e74c3c']

for rho, color in zip([0.0, 0.2, 0.5, 0.7, 1.0], colors_rho):
    ens_var = rho * sigma2 + (1 - rho) * sigma2 / B_values
    ax.plot(B_values, ens_var, color=color, linewidth=2.2,
            label=f'ρ = {rho:.1f}')
    ax.axhline(rho * sigma2, color=color, linestyle='--', alpha=0.35, linewidth=1)

ax.set_xlabel('Number of Trees (B)', fontsize=12)
ax.set_ylabel('Ensemble Variance (normalised)', fontsize=12)
ax.set_title('Ensemble Variance vs B for Different Tree Correlations (ρ)\n'
             'Dashed lines = irreducible floor (ρ·σ²)', fontsize=12, fontweight='bold')
ax.legend(title='Pairwise correlation ρ', fontsize=10)
ax.set_xlim(0, 200)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print('Key insight: With ρ=1 (identical trees), adding more trees does NOTHING.')
print('With ρ=0 (uncorrelated), variance → 0 as B → ∞.')
print('Bootstrap sampling lowers ρ, raising the ceiling on ensemble benefit.')

In [ ]:
# ============================================================
# Cell 14: Complete Performance Summary
# ============================================================
results = {
    'Single Tree (full depth)': rmse_single,
    'Manual Bagging (10 trees)': rmse_bag10,
    'Manual Bagging (100 trees)': rmse_bag100,
    'sklearn BaggingRegressor (100)': rmse_sklearn_bag,
    'Stacking (Tree+Ridge+kNN → Ridge)': rmse_stack,
}

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = ['#e74c3c', '#e67e22', '#f39c12', '#27ae60', '#8e44ad']
bars = ax.barh(list(results.keys()), list(results.values()),
               color=colors_bar, edgecolor='white', height=0.6)

for bar, val in zip(bars, results.values()):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}$k', va='center', fontsize=10)

ax.set_xlabel('Test RMSE ($k)', fontsize=12)
ax.set_title('All Methods — Test RMSE Comparison\n(Lower is Better)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

baseline = rmse_single
print('Method                                RMSE     Improvement vs Single Tree')
print('-' * 70)
for name, r in results.items():
    pct = 100*(baseline - r)/baseline
    print(f'{name:<40} {r:>5.1f}$k   {pct:+.1f}%')

## Key Takeaways

1. **Diversity drives ensemble benefit.** If models make the same mistakes, averaging them accomplishes nothing. Bootstrap sampling, different algorithms, or random feature subsets all create diversity.

2. **Bagging attacks variance.** It is most beneficial when the base model has high variance (e.g., deep decision trees). It does not help much with a high-bias base model (e.g., a shallow tree or linear model with underfitting).

3. **Boosting attacks bias.** It is most beneficial when the base model slightly underfits. Each round reduces the residuals, making the ensemble progressively more accurate.

4. **Stacking learns to combine.** Rather than fixing a combination rule (average), stacking learns from data which base models to trust for which patterns.

5. **The ensemble variance floor is $\rho \sigma^2$.** Adding more trees in bagging never increases variance, but returns diminish quickly. The floor is set by tree *correlation*, not by the count of trees.

6. **Stacking requires out-of-fold predictions** to avoid data leakage. If you train the meta-learner on the base models' predictions on their own training data, the base models will "look too good" (they memorised the labels), and the meta-learner learns to trust over-confident base models.

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:**
You bag 100 identical, perfectly correlated models — same training data, no randomness whatsoever. Does this improve performance over a single model? Why or why not? Ground your answer in the ensemble variance formula.

<details>
<summary>Show Answer</summary>

**No, it does not improve performance.**

The ensemble variance formula is:
$$\text{Var}_{\text{ensemble}} = \rho \sigma^2 + \frac{(1-\rho)\sigma^2}{B}$$

When models are perfectly correlated, $\rho = 1$, so:
$$\text{Var}_{\text{ensemble}} = 1 \cdot \sigma^2 + \frac{0 \cdot \sigma^2}{B} = \sigma^2$$

The ensemble variance equals the single-model variance, regardless of how many models $B$ you use. Averaging identical predictions gives you the same prediction. The averaging only helps when the errors are *different* (i.e., uncorrelated) — then the $(1-\rho)/B$ term shrinks with $B$. Diversity is a precondition, not an afterthought.
</details>

---

**Question 2:**
Boosting trains models sequentially — you cannot parallelise the training. Bagging can train all trees in parallel and is therefore much faster to train. What advantage does boosting offer that justifies its sequential (and thus slower) nature?

<details>
<summary>Show Answer</summary>

**Boosting reduces bias; bagging does not.**

Bagging averages many high-variance models. The average has the same bias as a single model — if a single decision tree systematically under-prices luxury homes, the bag of 100 trees will *still* under-price luxury homes (the bias is not cancelled by averaging, only the variance is).

Boosting addresses the residuals directly. Round 1 fits the data. Round 2 fits the *errors* of round 1. Round 3 fits the errors of rounds 1+2. Each new model is explicitly tasked with fixing what previous models got wrong. The ensemble bias decreases with each round.

Therefore, **if your problem has a high-bias base model** (e.g., shallow trees), boosting is dramatically more effective than bagging. The sequential training cost is justified because you are reducing a fundamentally different type of error. In practice, XGBoost and LightGBM use shallow trees (low variance, high bias) as base learners — the very setting where boosting excels.
</details>

---

**Question 3:**
Stacking uses predictions from base models as features for the meta-learner. Why must you use *out-of-fold* predictions (not the training set predictions) for the meta-learner's training data?

<details>
<summary>Show Answer</summary>

**To prevent data leakage and overfitting.**

If you train a base model on the full training set and then ask it to predict on that *same* training set, it produces near-perfect predictions (the model has memorised the labels). These over-optimistic predictions are what the meta-learner then uses as features.

The meta-learner learns: "When the tree predicts perfectly, trust it completely." But at test time, the base models have *not* seen the test data — so their predictions are noisier and less perfect. The meta-learner's learned trust is misplaced, and generalisation is poor.

**Out-of-fold (OOF) predictions** are generated by training on a subset of the training data and predicting on the held-out fold — data the model has *never* seen. These predictions have the same noise characteristics as genuine test-set predictions. The meta-learner therefore trains on a realistic picture of base model reliability, and its learned blending generalises properly.

In short: use OOF predictions to ensure the meta-learner trains on the same distribution it will see at inference time.
</details>

## Practical Guidelines

| Situation | Recommendation |
|---|---|
| Base model already has low bias | Prefer **Bagging** — reduce its variance |
| Base model underfits (high bias) | Prefer **Boosting** — reduce its bias |
| You have diverse model types | Try **Stacking** — exploit model diversity |
| Training time is critical | **Bagging** (parallelisable) |
| Maximum predictive accuracy needed | **Stacking** or **Boosting** |
| Interpretability matters | None of the above — use a single shallow tree or linear model |

**Next notebook:** Random Forests — the most successful bagging algorithm, augmented with random feature subsets to further decorrelate trees.

In [ ]:
# ============================================================
# Cell 15: Bonus — Visualise Boosting Intuition
#   (Preview for next topic: each round fits the residuals)
# ============================================================
from sklearn.ensemble import GradientBoostingRegressor

n_rounds_list = [1, 5, 20, 50, 100]
boost_rmses = []

for n_rounds in n_rounds_list:
    gb = GradientBoostingRegressor(
        n_estimators=n_rounds, max_depth=2, learning_rate=0.1, random_state=RANDOM_STATE
    )
    gb.fit(X_train, y_train)
    boost_rmses.append(rmse(y_test, gb.predict(X_test)))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_rounds_list, boost_rmses, marker='s', color='#c0392b',
        linewidth=2.5, markersize=7, label='Gradient Boosting RMSE')
ax.axhline(rmse_bag100, color='#27ae60', linestyle='--', linewidth=1.8,
           label=f'Bagging 100 trees = {rmse_bag100:.1f}$k')
ax.set_xlabel('Boosting Rounds', fontsize=11)
ax.set_ylabel('Test RMSE ($k)', fontsize=11)
ax.set_title('Boosting Rounds vs RMSE — Preview\n'
             'Boosting reduces bias with each round', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
for x, y_val in zip(n_rounds_list, boost_rmses):
    ax.annotate(f'{y_val:.1f}', (x, y_val), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Boosting (gradient, shallow trees) quickly surpasses bagging.')
print('The sequential, bias-reducing nature pays off on this dataset.')
print('We will cover Gradient Boosting in depth in Notebooks 7 & 8.')

In [ ]:
# ============================================================
# Cell 16: Final Summary Print
# ============================================================
print('=' * 60)
print('NOTEBOOK 5 COMPLETE: Ensemble Methods Overview')
print('=' * 60)
print()
print('Concepts covered:')
print('  1. Bagging (Bootstrap Aggregating) — reduces variance')
print('  2. Manual bagging from scratch')
print('  3. Variance reduction demonstration (50 trials)')
print('  4. Stacking with out-of-fold predictions — reduces both')
print('  5. sklearn BaggingRegressor vs manual implementation')
print('  6. Diversity: why correlated models give no benefit')
print('  7. Learning curves: n_estimators 1 → 200')
print('  8. Bias-variance formula visualisation')
print('  9. Boosting preview (bias reduction mechanism)')
print()
print('Next: Notebook 6 — Random Forests')
print('  Bagging + random feature subsets = decorrelated trees')